# CrispEmbed — LAYOUT_CONV_F16 tensor-core (T4) verdict

**Purpose (PLAN.md round-N+4 queue #4):** `LAYOUT_CONV_F16` loses on M1 Metal and is
time-neutral-with-region-drift on P100. The only open question is **tensor-core hardware**
(T4, compute 7.5): does the f16 conv path win there, and does the known 20→19 region drift
reproduce? Kaggle served P100 seven times across two accounts and two days, so this runs on
Colab, whose free tier serves T4.

**How to run:**
1. `Runtime → Change runtime type → T4 GPU` (any GPU works; the log records the draw honestly —
   but only a T4 answers the question).
2. `Runtime → Run all`. The build is cold (~10–20 min); the arms take ~2 min after that.
3. Read the `SUMMARY` block at the end of the last cell (also saved to `/content/t4draw.log`).
   Paste that block back into the CrispEmbed session/PLAN.

**Verdict rules (proof-of-work):** a timing row without its matching region output is a FAIL,
never a win. The second `Phase 1` line per arm is the warm one — compare those. If regions
differ, the first differing line is printed; f16 stays gated unless it wins time on T4 with
IDENTICAL regions.

In [ ]:
import subprocess
gpu = subprocess.run("nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "none"
print(f"GPU: {gpu}")
print("T4-DRAW: " + ("YES - tensor-core arm is decisive" if "T4" in gpu
                     else "NO - not a T4; result will be recorded but does not close the question"))
if "T4" not in gpu:
    print("Tip: Runtime -> Change runtime type -> T4 GPU, then Run all again.")

In [ ]:
%%shell
set -eo pipefail
exec > >(tee /content/t4draw.log) 2>&1

echo "== toolchain =="
pip install -q ninja huggingface_hub hf_transfer
apt-get install -y -qq ccache > /dev/null || true

echo "== clone =="
cd /content
if [ ! -d CrispEmbed ]; then
  git clone --depth 1 --recursive https://github.com/CrispStrobe/CrispEmbed.git
fi
cd CrispEmbed && mkdir -p build

ARCH=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d '.')
echo "CUDA arch: $ARCH"

echo "== configure =="
cmake -G Ninja -B build -DCMAKE_BUILD_TYPE=Release \
  -DGGML_CUDA=ON -DGGML_CUDA_NO_VMM=ON -DCMAKE_CUDA_ARCHITECTURES=$ARCH \
  -DCMAKE_C_COMPILER_LAUNCHER=ccache -DCMAKE_CXX_COMPILER_LAUNCHER=ccache \
  -DCMAKE_CUDA_COMPILER=/usr/local/cuda/bin/nvcc

echo "== build (cold ~10-20 min on T4) =="
cmake --build build -j$(nproc) --target crispembed-cli 2>&1 | tail -3

echo "== model =="
python -c "from huggingface_hub import hf_hub_download; print(hf_hub_download('cstr/layout-heron-gguf','layout-heron-f32.gguf',local_dir='/content/models'))"

set +e  # arm failures must print their rc, not kill the cell
echo "== arms =="
BIN=/content/CrispEmbed/build/crispembed
[ -f $BIN ] || BIN=/content/CrispEmbed/build/bin/crispembed
IMG=/content/CrispEmbed/tests/regression/images/scan_page_pd.png
M=/content/models/layout-heron-f32.gguf
GPU=$(nvidia-smi --query-gpu=name --format=csv,noheader | head -1)

for ARM in f32 f16; do
  if [ "$ARM" = "f16" ]; then export LAYOUT_CONV_F16=1; else unset LAYOUT_CONV_F16; fi
  CRISPEMBED_LAYOUT_DETECT_BENCH=1 CRISPEMBED_LAYOUT_REPEAT=2 \
    $BIN -m $M --layout $IMG -t 4 > /content/layout_$ARM.txt 2> /content/layout_$ARM.err
  RC=$?
  echo "RESULT| layout.$ARM rc=$RC stdout_bytes=$(wc -c < /content/layout_$ARM.txt)"
  grep "Phase 1" /content/layout_$ARM.err | sed "s/^/RESULT| layout.$ARM [$GPU]: /"
done

echo ""
echo "========================================================================"
echo "SUMMARY (GPU: $GPU)"
echo "========================================================================"
grep -h "RESULT|" /content/t4draw.log || true  # file order = cold rep then warm rep per arm
if cmp -s /content/layout_f32.txt /content/layout_f16.txt; then
  echo "layout regions identical on $GPU: True"
else
  echo "layout regions identical on $GPU: False"
  echo "line counts: f32=$(wc -l < /content/layout_f32.txt) f16=$(wc -l < /content/layout_f16.txt)"
  diff /content/layout_f32.txt /content/layout_f16.txt | head -6
fi
echo done